In [1]:
# importing the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

In [2]:
injection_df = pd.read_csv('./injection_only.csv')
injection_df.columns = ['AOA', 'injection_location', 'injection_slot_size', 'cl', 'cd', 'Re', 'chord_length']
injection_df

,AOA,injection_location,injection_slot_size,cl,cd,Re,chord_length
0,-20.0,4,0.5,-0.65,0.280,1000000,460
1,-19.5,4,0.5,-0.66,0.265,1000000,460
2,-19.0,4,0.5,-0.67,0.250,1000000,460
3,-18.5,4,0.5,-0.68,0.235,1000000,460
4,-18.0,4,0.5,-0.69,0.220,1000000,460
...,...,...,...,...,...,...,...
679,25.5,50,2.0,1.77,1.052,2000000,600
680,26.0,50,2.0,1.74,1.084,2000000,600
681,26.5,50,2.0,1.71,1.116,2000000,600
682,27.0,50,2.0,1.68,1.148,2000000,600


In [4]:
# setting up the data
data_df = injection_df.drop(columns=['cl', 'cd'])
data_df
injection_data = np.array(data_df)
injection_data

array([[-2.00e+01,  4.00e+00,  5.00e-01,  1.00e+06,  4.60e+02],
       [-1.95e+01,  4.00e+00,  5.00e-01,  1.00e+06,  4.60e+02],
       [-1.90e+01,  4.00e+00,  5.00e-01,  1.00e+06,  4.60e+02],
       ...,
       [ 2.65e+01,  5.00e+01,  2.00e+00,  2.00e+06,  6.00e+02],
       [ 2.70e+01,  5.00e+01,  2.00e+00,  2.00e+06,  6.00e+02],
       [ 2.75e+01,  5.00e+01,  2.00e+00,  2.00e+06,  6.00e+02]],
      shape=(684, 5))

In [10]:
#setting up the targets
injection_cl = np.array(injection_df['cl'])
injection_cd = np.array(injection_df['cd'])
# injection_cl

In [7]:
# splitting the data for cl and and cd separately
from sklearn.model_selection import train_test_split

X_train_cl, X_test_cl, y_train_cl, y_test_cl = train_test_split(injection_data, injection_cl, random_state=0)
X_train_cd, X_test_cd, y_train_cd, y_test_cd = train_test_split(injection_data, injection_cd, random_state=0)

# X_train_cd
# X_train_cl

### Data Scaling

In [8]:
# scaling the data
from sklearn.preprocessing import StandardScaler

scaler_cl = StandardScaler().fit(X_train_cl)
scaler_cd = StandardScaler().fit(X_train_cd)
# scaling the cl data

X_train_cl_scaled = scaler_cl.transform(X_train_cl)
X_test_cl_scaled = scaler_cl.transform(X_test_cd)

# scaling the cd data
X_train_cd_scaled = scaler_cd.transform(X_train_cd)
X_test_cd_scaled = scaler_cd.transform(X_test_cd)

# X_train_cd_scaled
# X_train_cl_scaled


### Training the models

In [9]:
from sklearn.neural_network import MLPRegressor

mlp_cl = MLPRegressor(random_state=0,hidden_layer_sizes=(100, 100), max_iter=600).fit(X_train_cl_scaled, y_train_cl)

mlp_cd = MLPRegressor(random_state=0, early_stopping=True, hidden_layer_sizes=(100, 100), max_iter=250).fit(X_train_cd_scaled, y_train_cd)

print("CL prediction accuracy")
print(f"Cl training score: {mlp_cl.score(X_train_cl_scaled, y_train_cl)}")
print(f"Cl Test score: {mlp_cl.score(X_test_cl_scaled, y_test_cl)}")


print("\nCD prediction accuracy")
print(f"Cd training score: {mlp_cd.score(X_train_cd_scaled, y_train_cd)}")
print(f"Cd Test score: {mlp_cd.score(X_test_cd_scaled, y_test_cd)}")



CL prediction accuracy
Cl training score: 0.9929073000411027
Cl Test score: 0.9940123017339335

CD prediction accuracy
Cd training score: 0.9965392073241033
Cd Test score: 0.9956777128647661


In [ ]:
mlp_cd.predict

In [25]:
all_geometries

[(5, 0.5),
 (5, 1),
 (5, 1.5),
 (5, 2),
 (5, 2.5),
 (10, 0.5),
 (10, 1),
 (10, 1.5),
 (10, 2),
 (10, 2.5),
 (15, 0.5),
 (15, 1),
 (15, 1.5),
 (15, 2),
 (15, 2.5),
 (20, 0.5),
 (20, 1),
 (20, 1.5),
 (20, 2),
 (20, 2.5),
 (25, 0.5),
 (25, 1),
 (25, 1.5),
 (25, 2),
 (25, 2.5),
 (30, 0.5),
 (30, 1),
 (30, 1.5),
 (30, 2),
 (30, 2.5),
 (35, 0.5),
 (35, 1),
 (35, 1.5),
 (35, 2),
 (35, 2.5),
 (40, 0.5),
 (40, 1),
 (40, 1.5),
 (40, 2),
 (40, 2.5),
 (45, 0.5),
 (45, 1),
 (45, 1.5),
 (45, 2),
 (45, 2.5),
 (50, 0.5),
 (50, 1),
 (50, 1.5),
 (50, 2),
 (50, 2.5)]

### Predicting for all the combinations of injection locations and slot sizes

In [35]:
import itertools

injection_locations = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
injection_slot_sizes = [0.5, 1, 1.5, 2, 2.5]
all_geometries = list(itertools.product(
    injection_locations,
    injection_slot_sizes
))
angles_of_attack = np.arange(0, 29, 1)
Re = 2000000
chord_length = 600
output_dict = {
    "AOA" : [],
    "injection_location" : [],
    "injection_slot_size" : [],
    "predicted_cl" : [],
    "predicted_cd" : []
}
for i, geometry in enumerate(all_geometries):
    print(f"Computing for geometry #{i}.......")
    injection_location = geometry[0]
    injection_slot_size = geometry[1]
    for aoa in angles_of_attack:
        # get the results
        print(f"{aoa}", end=',')
        aoa = int(aoa)
        X_input = [[aoa, injection_location, injection_slot_size, Re, chord_length]]
        X_input_scaled = scaler_cl.transform(X_input)
        predicted_cl = float(mlp_cl.predict(X_input_scaled)[0])
        predicted_cd = float(mlp_cd.predict(X_input_scaled)[0])
        
        # store the results
        output_dict["AOA"].append(aoa)
        output_dict['injection_location'].append(injection_location)
        output_dict['injection_slot_size'].append(injection_slot_size)
        output_dict['predicted_cl'].append(predicted_cl)
        output_dict['predicted_cd'].append(predicted_cd)
    print(f"\nGeometry {i} complete! ")
        

Computing for geometry #0.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,
Geometry 0 complete! 
Computing for geometry #1.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,
Geometry 1 complete! 
Computing for geometry #2.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,
Geometry 2 complete! 
Computing for geometry #3.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,
Geometry 3 complete! 
Computing for geometry #4.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,
Geometry 4 complete! 
Computing for geometry #5.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,
Geometry 5 complete! 
Computing for geometry #6.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,
Geometry 6 complete! 
Computing for geometry #7.......
0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,1

In [ ]:
output_dict

In [40]:
output_df = pd.DataFrame(output_dict, index=None)
output_df

,AOA,injection_location,injection_slot_size,predicted_cl,predicted_cd
0,0,5,0.5,-0.069841,0.324688
1,1,5,0.5,-0.022410,0.340215
2,2,5,0.5,0.023313,0.354855
3,3,5,0.5,0.069036,0.370730
4,4,5,0.5,0.114899,0.387785
...,...,...,...,...,...
1445,24,50,2.5,1.708400,0.780630
1446,25,50,2.5,1.732484,0.822954
1447,26,50,2.5,1.748324,0.867232
1448,27,50,2.5,1.761923,0.913461


In [42]:
output_df.to_csv('injection_output_data.csv', index=False)